In [459]:
import torch
import torch.nn.functional as F
import numpy as np

def gradient_check(model: torch.nn.Module,
                   image: torch.Tensor,
                   roi_pixels: list[int],
                   label_idx: int,
                   perturbation_strength: float = 1.0):
    """
    Performs a gradient check by comparing gradients with and without perturbation.

    :param model: Your trained model, expected to be in eval() mode externally.
    :param image: Input image tensor of shape (1, C, H, W). 
                  Will be set requires_grad=True for gradient computation.
    :param roi_pixels: Flattened pixel indices to perturb.
    :param label_idx: Class index for gradient calculation.
    :param perturbation_strength: Magnitude of random perturbation added to each pixel.
    :return: (original_grad, perturbed_grad, grad_difference)
             original_grad, perturbed_grad are shape (1, C, H, W),
             grad_difference is a scalar (torch.Tensor).
    """

    model.zero_grad(set_to_none=True)

    # Ensure we can compute grads on 'image'
    image = image.detach().clone()  # so we don't mutate the original
    image.requires_grad_(True)

    # Forward pass (original image)
    logits = model(image)          # shape: (1, num_classes)
    prob_original = F.softmax(logits, dim=1)[0, label_idx]
    prob_original.backward()

    # Clone the gradient for the original image
    original_grad = image.grad.clone()

    # Reset gradient on the image so we start fresh for the next pass
    image.grad.zero_()

    # Create perturbed version of the image
    # It's often safer to manipulate it in a no_grad() block
    with torch.no_grad():
        perturbed_image = image.clone().detach()
        _, C, H, W = perturbed_image.shape

        # Perturb each pixel in roi_pixels
        for pixel in roi_pixels:
            c, h, w = np.unravel_index(pixel, (C, H, W))
            noise = torch.randn(1).item()  # ~ N(0,1)
            perturbed_image[0, c, h, w] += perturbation_strength * noise

    perturbed_image.requires_grad_(True)
    model.zero_grad(set_to_none=True)

    # Forward pass (perturbed image)
    logits_perturbed = model(perturbed_image)
    prob_perturbed = F.softmax(logits_perturbed, dim=1)[0, label_idx]
    prob_perturbed.backward()

    # Clone gradient of perturbed image
    perturbed_grad = perturbed_image.grad.clone()

    # Compute mean absolute difference of gradients
    grad_difference = (perturbed_grad - original_grad).abs().mean()

    pertube_norm = torch.norm(perturbed_grad, p=2) **2
    original_norm = torch.norm(original_grad, p=2) **2

    grad_norm_diffrence = pertube_norm - original_norm

    # Log info
    print("Original Probability:", prob_original.item())
    print("Perturbed Probability:", prob_perturbed.item())
    print("Mean Absolute Gradient Difference:", grad_difference.item())

    print("Norm of Original Gradient:", original_norm.item())
    print("Norm of Perturbed Gradient:", pertube_norm.item())
    print("Gradient Norm Difference:", grad_norm_diffrence.item())

    return original_grad, perturbed_grad, grad_difference


In [460]:
from importance import get_model_labels, load_model, prepare_image, get_labels
from vargrad import load_attr_maps_to_tensors, get_top_influential_flat_indexes
import torch
import random

PATH = "checkpoints/checkpoint_86epoch_0.9327trainF1_0.9344valF1_4c.pth"

#IMAGE_PATH = "images2explain/Horse_Zebra.png"
IMAGE_PATH = "images2explain/Giraffe_Lion.png" 
#IMAGE_PATH = "images2explain/Zebra_Lion.png"
#IMAGE_PATH = "images2explain/Lion_Horse.png"

labels = get_model_labels()  
model = load_model(PATH, labels) 
img, img_tensor = prepare_image(IMAGE_PATH)
gt_labels = get_labels(IMAGE_PATH)

attr_maps_dir = "individual_vargrad_maps_grayscale"
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

attr_tensors = load_attr_maps_to_tensors(attr_maps_dir, image_size=(224, 224), device=device)

for label, tensor in attr_tensors.items():
    print(f"{label}: shape={tensor.shape}, device={tensor.device}")

top_indices_dict = get_top_influential_flat_indexes(attr_tensors, top_k=1000)
no_info_list = list(random.sample(range(0, 224 * 224), 1000))

Lion: shape=torch.Size([1, 224, 224]), device=mps:0
Giraffe: shape=torch.Size([1, 224, 224]), device=mps:0


In [461]:
label = "Lion"
class_idx = labels.index(label)

original_grad, perturbed_grad, diff = gradient_check(
    model=model,
    image=img_tensor,
    roi_pixels=top_indices_dict["Giraffe"][:10], 
    label_idx=class_idx,
    perturbation_strength=1.0
)
print()
original_grad, perturbed_grad, diff = gradient_check(
    model=model,
    image=img_tensor,
    roi_pixels=top_indices_dict["Lion"][:10], 
    label_idx=class_idx,
    perturbation_strength=1.0
)
print()
original_grad, perturbed_grad, diff = gradient_check(
    model=model,
    image=img_tensor,
    roi_pixels=no_info_list, 
    label_idx=class_idx,
    perturbation_strength=1.0
)

Original Probability: 0.5552263259887695
Perturbed Probability: 0.5957571268081665
Mean Absolute Gradient Difference: 0.0008556258981116116
Norm of Original Gradient: 4.081401824951172
Norm of Perturbed Gradient: 4.7994842529296875
Gradient Norm Difference: 0.7180824279785156

Original Probability: 0.5552263259887695
Perturbed Probability: 0.5961703658103943
Mean Absolute Gradient Difference: 0.000836381979752332
Norm of Original Gradient: 4.081401824951172
Norm of Perturbed Gradient: 4.436119556427002
Gradient Norm Difference: 0.3547177314758301

Original Probability: 0.5552263259887695
Perturbed Probability: 0.22249792516231537
Mean Absolute Gradient Difference: 0.0034197864588350058
Norm of Original Gradient: 4.081401824951172
Norm of Perturbed Gradient: 1.6930032968521118
Gradient Norm Difference: -2.3883986473083496


In [462]:
label = "Giraffe"
class_idx = labels.index(label)

original_grad, perturbed_grad, diff = gradient_check(
    model=model,
    image=img_tensor,
    roi_pixels=top_indices_dict["Lion"][:10],
    label_idx=class_idx,
    perturbation_strength=1.0
)
print()
original_grad, perturbed_grad, diff = gradient_check(
    model=model,
    image=img_tensor,
    roi_pixels=top_indices_dict["Giraffe"][:10], 
    label_idx=class_idx,
    perturbation_strength=1.0
)
print()
original_grad, perturbed_grad, diff = gradient_check(
    model=model,
    image=img_tensor,
    roi_pixels=no_info_list, 
    label_idx=class_idx,
    perturbation_strength=1.0
)

Original Probability: 0.4238588809967041
Perturbed Probability: 0.3693337142467499
Mean Absolute Gradient Difference: 0.0007577176438644528
Norm of Original Gradient: 4.186825275421143
Norm of Perturbed Gradient: 4.222098350524902
Gradient Norm Difference: 0.035273075103759766

Original Probability: 0.4238588809967041
Perturbed Probability: 0.4211297929286957
Mean Absolute Gradient Difference: 0.0007695057429373264
Norm of Original Gradient: 4.186825275421143
Norm of Perturbed Gradient: 5.048892021179199
Gradient Norm Difference: 0.8620667457580566

Original Probability: 0.4238588809967041
Perturbed Probability: 0.6947486996650696
Mean Absolute Gradient Difference: 0.003832087153568864
Norm of Original Gradient: 4.186825275421143
Norm of Perturbed Gradient: 2.755772113800049
Gradient Norm Difference: -1.4310531616210938


In [463]:
from importance import compute_per_sample_gradient

label_idx = labels.index("Lion")

# vmap version
img_tensor.requires_grad_(True)
batch_grad_vmap = compute_per_sample_gradient(model, img_tensor, label_idx)
print("Batch gradient using vmap:", batch_grad_vmap.shape)

img_tensor.requires_grad_(True)
logits = model(img_tensor)
prob = F.softmax(logits, dim=1)[:, label_idx].sum()  # sum so we can .backward once for the whole batch
prob.backward()
batch_grad_backward = img_tensor.grad.clone()
img_tensor.grad.zero_()
print("Batch gradient using backward:", batch_grad_backward.shape)

diff = (batch_grad_vmap - batch_grad_backward).abs().mean()
print("Mean difference between vmap gradient and backward gradient:", diff.item())


Batch gradient using vmap: torch.Size([1, 3, 224, 224])
Batch gradient using backward: torch.Size([1, 3, 224, 224])
Mean difference between vmap gradient and backward gradient: 6.304580438154517e-06


In [464]:
import torch
import torch.nn.functional as F
import numpy as np
from importance import compute_per_sample_gradient
from regularization import Perturbation

def check_gradients_loop_vs_vmap(model, images, label_idx):
    """
    Compare per-sample gradients computed via:
      1) vmap-based approach (compute_per_sample_gradient)
      2) manual per-image loop with .backward()
      
    Arguments:
      model: your neural network model
      images: tensor of shape (B, C, H, W) - the input batch
      label_idx: class index for gradient calculation
      compute_per_sample_gradient: function that uses vmap to compute gradient
    
    Returns:
      batch_grad_vmap: shape (B, C, H, W)
      batch_grad_loop: shape (B, C, H, W)
      diff: scalar mean absolute difference
    """
    model.eval()  # ensure model is in eval mode

    # ----------- 1) vmap-based gradients -----------
    # Make a clone so we don't accidentally mess up original data
    images_vmap = images.clone().requires_grad_(True)
    batch_grad_vmap = compute_per_sample_gradient(model, images_vmap, label_idx)
    # shape: (B, C, H, W)

    # ----------- 2) Loop-based gradients -----------
    B = images.size(0)
    manual_grads = []

    for i in range(B):
        # Create a fresh tensor for this single image
        single_img = images[i].unsqueeze(0).clone().detach()  # shape (1, C, H, W)
        single_img.requires_grad_(True)

        # Forward pass for this 1-image batch
        logits_i = model(single_img)  # shape: (1, num_classes)
        prob_i = F.softmax(logits_i, dim=1)[0, label_idx]  # scalar for chosen label

        # Backward pass to get gradient w.r.t. single_img
        prob_i.backward()

        # single_img.grad has shape (1, C, H, W)
        grad_i = single_img.grad[0].clone()  # shape: (C, H, W)
        manual_grads.append(grad_i)

    # Stack them so shape matches (B, C, H, W)
    batch_grad_loop = torch.stack(manual_grads, dim=0)

    # ----------- 3) Compare -----------
    diff = (batch_grad_vmap - batch_grad_loop).abs().mean()

    print(f"vmap gradient shape: {batch_grad_vmap.shape}")
    print(f"loop gradient shape: {batch_grad_loop.shape}")
    print(f"Mean difference between vmap and loop: {diff.item():.6f}")

    return batch_grad_vmap, batch_grad_loop, diff


# 1) Suppose 'images' is your batch from 'perturb_tensor_subset(...)'
images = Perturbation.perturb_tensor_subset(img_tensor, top_indices_dict["Lion"][:3], n_samples=3, perturbation=True, num_channels=3)
# shape = (B, C, H, W)

# 2) label_idx is the target class index
label_idx = labels.index("Lion")

# 3) Then simply call:
vmap_grad, loop_grad, difference = check_gradients_loop_vs_vmap(model, images, label_idx)


vmap gradient shape: torch.Size([9, 3, 224, 224])
loop gradient shape: torch.Size([9, 3, 224, 224])
Mean difference between vmap and loop: 0.000002


In [ ]:
import torch
import torch.nn.functional as F
from importance import compute_per_sample_gradient
from regularization import Perturbation

def extended_check_gradients_and_sqnorm(
    model: torch.nn.Module,
    images: torch.Tensor,
    label_idx: int,
    pixel_list: list[int],
    n_samples: int
):
    """
    1) Compare per-sample gradients computed via:
       - vmap-based approach (compute_per_sample_gradient)
       - manual per-image loop with .backward()
    2) Compute the average squared gradient-norm for each pixel (in pixel_list)
       by grouping the batch into sets of n_samples.

    Args:
        model: your trained model (should be in eval mode outside).
        images: shape (B, C, H, W). B == len(pixel_list)*n_samples
        label_idx: class index for gradient computation
        pixel_list: the list of pixel indices used for perturbation
        n_samples: how many perturbations per pixel were generated

    Returns:
        batch_grad_vmap: shape (B, C, H, W)
        batch_grad_loop: shape (B, C, H, W)
        diff: scalar, mean absolute difference between vmap and loop gradients
        avg_sqnorm_per_pixel_vmap: dict {pixel_idx -> float} 
            average squared gradient-norm for each pixel’s sub-batch (vmap-based)
        avg_sqnorm_per_pixel_loop: dict {pixel_idx -> float}
            same, but from the loop-based gradients
    """
    model.eval()

    # ----------------------------------------------------------------
    # 1) Vmap-based gradients
    # ----------------------------------------------------------------
    images_vmap = images.clone().requires_grad_(True)
    batch_grad_vmap = compute_per_sample_gradient(model, images_vmap, label_idx)
    # shape: (B, C, H, W)

    # ----------------------------------------------------------------
    # 2) Loop-based gradients (one sample at a time)
    # ----------------------------------------------------------------
    B = images.size(0)
    manual_grads = []

    for i in range(B):
        single_img = images[i].unsqueeze(0).clone().detach()
        single_img.requires_grad_(True)

        logits_i = model(single_img)
        prob_i = F.softmax(logits_i, dim=1)[0, label_idx]
        prob_i.backward()

        grad_i = single_img.grad[0].clone()  # shape: (C, H, W)
        manual_grads.append(grad_i)

    batch_grad_loop = torch.stack(manual_grads, dim=0)  # shape: (B, C, H, W)

    # ----------------------------------------------------------------
    # 3) Compare difference (vmap vs loop)
    # ----------------------------------------------------------------
    diff = (batch_grad_vmap - batch_grad_loop).abs().mean()

    #print(f"vmap gradient shape: {batch_grad_vmap.shape}")
    #print(f"loop gradient shape: {batch_grad_loop.shape}")
    print(f"Mean difference between vmap and loop: {diff.item():.6f}")

    # ----------------------------------------------------------------
    # 4) Compute average squared grad-norm per pixel
    #    by grouping sub-batches of size n_samples
    # ----------------------------------------------------------------
    avg_sqnorm_per_pixel_vmap = {}
    avg_sqnorm_per_pixel_loop = {}

    for idx_pixel, pix in enumerate(pixel_list):
        start = idx_pixel * n_samples
        end = (idx_pixel + 1) * n_samples

        # Extract the relevant sub-batch
        subbatch_vmap = batch_grad_vmap[start:end]   # shape: (n_samples, C, H, W)
        subbatch_loop = batch_grad_loop[start:end]

        # Compute squared norms for each sample
        # e.g. L2-norm: sum of squares across (C,H,W)
        sqnorm_vmap = (subbatch_vmap ** 2).sum(dim=(1,2,3))  # shape: (n_samples,)
        sqnorm_loop = (subbatch_loop ** 2).sum(dim=(1,2,3))  # shape: (n_samples,)

        # Average over n_samples
        avg_sqnorm_vmap = sqnorm_vmap.mean().item()
        avg_sqnorm_loop = sqnorm_loop.mean().item()

        avg_sqnorm_per_pixel_vmap[pix] = avg_sqnorm_vmap
        avg_sqnorm_per_pixel_loop[pix] = avg_sqnorm_loop

    return (
        batch_grad_vmap,
        batch_grad_loop,
        diff,
        avg_sqnorm_per_pixel_vmap,
        avg_sqnorm_per_pixel_loop,
    )

In [466]:
n_samples=20
pixel_list = top_indices_dict["Lion"][:10]
images_perturbed = Perturbation.perturb_tensor_subset(
     img_tensor, pixel_list, n_samples=n_samples, perturbation=True, num_channels=3
 )
# images_perturbed has shape (B, C, H, W), with B=9 in this example.

label_idx = labels.index("Lion")
results = extended_check_gradients_and_sqnorm(
     model, images_perturbed, label_idx, pixel_list, n_samples=n_samples
 )
#
# # Unpack results
vmap_grad, loop_grad, difference, avg_vmap, avg_loop = results
print("difference:", difference)
print("Pertube Lion pixels with Lion label")
for px in pixel_list:
    print(f"Pixel {px}, vmap avg sqnorm = {avg_vmap[px]}, loop avg sqnorm = {avg_loop[px]}")


label_idx = labels.index("Giraffe")
results = extended_check_gradients_and_sqnorm(
     model, images_perturbed, label_idx, pixel_list, n_samples=n_samples
 )
#
# # Unpack results
vmap_grad, loop_grad, difference, avg_vmap, avg_loop = results
print("difference:", difference)
print("Pertube Lion pixels with Giraffe label")
for px in pixel_list:
    print(f"Pixel {px}, vmap avg sqnorm = {avg_vmap[px]}, loop avg sqnorm = {avg_loop[px]}")    

Mean difference between vmap and loop: 0.000004
difference: tensor(3.8198e-06, grad_fn=<MeanBackward0>)
Pertube Lion pixels with Lion label
Pixel 34687, vmap avg sqnorm = 4.123682975769043, loop avg sqnorm = 4.125169277191162
Pixel 37119, vmap avg sqnorm = 4.459249019622803, loop avg sqnorm = 4.459638595581055
Pixel 34915, vmap avg sqnorm = 4.230334281921387, loop avg sqnorm = 4.231566429138184
Pixel 29955, vmap avg sqnorm = 4.211661338806152, loop avg sqnorm = 4.212873935699463
Pixel 11350, vmap avg sqnorm = 4.41945743560791, loop avg sqnorm = 4.419531345367432
Pixel 32639, vmap avg sqnorm = 4.095236301422119, loop avg sqnorm = 4.095728874206543
Pixel 38460, vmap avg sqnorm = 4.402082920074463, loop avg sqnorm = 4.4025750160217285
Pixel 34465, vmap avg sqnorm = 4.106295585632324, loop avg sqnorm = 4.10697078704834
Pixel 38467, vmap avg sqnorm = 4.291430473327637, loop avg sqnorm = 4.29217529296875
Pixel 38017, vmap avg sqnorm = 4.339288234710693, loop avg sqnorm = 4.339848041534424
Me

In [467]:
n_samples=20
pixel_list = top_indices_dict["Giraffe"][:10]
images_perturbed = Perturbation.perturb_tensor_subset(
     img_tensor, pixel_list, n_samples=n_samples, perturbation=True, num_channels=3
 )
# images_perturbed has shape (B, C, H, W), with B=9 in this example.

label_idx = labels.index("Lion")
results = extended_check_gradients_and_sqnorm(
     model, images_perturbed, label_idx, pixel_list, n_samples=n_samples
 )
#
# # Unpack results
vmap_grad, loop_grad, difference, avg_vmap, avg_loop = results
print("difference:", difference)
print("Pertube Giraffe pixels with Lion label")
for px in pixel_list:
    print(f"Pixel {px}, vmap avg sqnorm = {avg_vmap[px]}, loop avg sqnorm = {avg_loop[px]}")


label_idx = labels.index("Giraffe")
results = extended_check_gradients_and_sqnorm(
     model, images_perturbed, label_idx, pixel_list, n_samples=n_samples
 )
#
# # Unpack results
vmap_grad, loop_grad, difference, avg_vmap, avg_loop = results
print("difference:", difference)
print("Pertube Giraffe pixels with Giraffe label")
for px in pixel_list:
    print(f"Pixel {px}, vmap avg sqnorm = {avg_vmap[px]}, loop avg sqnorm = {avg_loop[px]}")    

Mean difference between vmap and loop: 0.000003
difference: tensor(3.4615e-06, grad_fn=<MeanBackward0>)
Pertube Giraffe pixels with Lion label
Pixel 10670, vmap avg sqnorm = 4.342875003814697, loop avg sqnorm = 4.34309196472168
Pixel 35125, vmap avg sqnorm = 4.189061641693115, loop avg sqnorm = 4.190349578857422
Pixel 8871, vmap avg sqnorm = 4.059965133666992, loop avg sqnorm = 4.060050010681152
Pixel 6174, vmap avg sqnorm = 4.483685493469238, loop avg sqnorm = 4.484403610229492
Pixel 10674, vmap avg sqnorm = 4.413110256195068, loop avg sqnorm = 4.413821220397949
Pixel 31095, vmap avg sqnorm = 4.175993919372559, loop avg sqnorm = 4.1763834953308105
Pixel 12021, vmap avg sqnorm = 4.430582523345947, loop avg sqnorm = 4.431015491485596
Pixel 36464, vmap avg sqnorm = 3.945913791656494, loop avg sqnorm = 3.946820020675659
Pixel 35800, vmap avg sqnorm = 4.116030216217041, loop avg sqnorm = 4.1165056228637695
Pixel 11573, vmap avg sqnorm = 4.3480730056762695, loop avg sqnorm = 4.3483319282531

In [469]:
img_tensor = torch.arange(0.0, 9.0).view(1,1,3,3)  # Dummy tensor for example

# 1) Suppose 'images' is your batch from 'perturb_tensor_subset(...)'
images = Perturbation.perturb_tensor_subset(img_tensor, [1,3], n_samples=3, perturbation=True, num_channels=1)
# shape = (B, C, H, W)

print(images)  # shape: (B, C, H, W)


tensor([[[[0.0000, 1.7901, 2.0000],
          [3.0000, 4.0000, 5.0000],
          [6.0000, 7.0000, 8.0000]]],


        [[[0.0000, 0.3100, 2.0000],
          [3.0000, 4.0000, 5.0000],
          [6.0000, 7.0000, 8.0000]]],


        [[[0.0000, 1.0734, 2.0000],
          [3.0000, 4.0000, 5.0000],
          [6.0000, 7.0000, 8.0000]]],


        [[[0.0000, 1.0000, 2.0000],
          [3.1801, 4.0000, 5.0000],
          [6.0000, 7.0000, 8.0000]]],


        [[[0.0000, 1.0000, 2.0000],
          [4.1455, 4.0000, 5.0000],
          [6.0000, 7.0000, 8.0000]]],


        [[[0.0000, 1.0000, 2.0000],
          [1.8089, 4.0000, 5.0000],
          [6.0000, 7.0000, 8.0000]]]])
